In [1]:
import os
import sys
sys.path.append(os.path.abspath("../.."))

from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.stores import InMemoryStore
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_classic.chains import HypotheticalDocumentEmbedder, LLMChain

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction


from src.backend.logger import GLOBAL_LOGGER as log
from src.backend.core.config import settings
from src.backend.rag.embeddings import get_embeddings


****Data Ingestion****

In [2]:
def load_document(directory_path):
    try:
        documents = []
        for filename in os.listdir(directory_path):
            file_path = os.path.join(directory_path, filename)
            if filename.endswith(".pdf"):
                loader = PyPDFLoader(file_path)
                documents.extend(loader.load())
            elif filename.endswith(".docx"):
                # First: extract normal docx text
                loader = Docx2txtLoader(file_path)
                documents.extend(loader.load())
            elif filename.endswith(".txt"):
                loader = TextLoader(file_path, encoding="utf-8")
                documents.extend(loader.load())

        return documents
    except Exception as e:
        log.error("Error loading documents from directory", error=str(e), directory=directory_path)
        raise e

docs = load_document("./TempData")
docs

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-12-14T08:45:46+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-12-14T08:45:46+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': './TempData\\Acme_FY2024_UltraDense_Report.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='ACME MANUFACTURING LTD – FY2024 CONSOLIDATED REPORT\nAcme Manufacturing Ltd is a multinational industrial manufacturing company operating across the United States, Germany, and the\nUnited Kingdom. The company manufactures heavy machinery, automotive components, and precision-engineered industrial\nequipment for aerospace and defense sectors. Acme’s customers include original equipment manufacturers, government agencies,\nand industrial distributors. The company prepares consolidated financial statements in accordance with International Financial\nReporting Standards 

****Embeddings****

In [3]:
embeddings = get_embeddings("OpenAI (text-embedding-3-small)")

****Create Croma Client****

In [4]:
# Initialize Chroma persistent client
_chroma_client = chromadb.PersistentClient(path="./chroma_db")

Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


****Vector Store****

In [5]:
vector_store = Chroma(
    collection_name="multi_doc_assist",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

****Adding Documents to Chroma Vector Store****

In [6]:
vector_store.add_documents(documents=docs)

HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


['3dfaaa83-b9cb-4d1f-8d38-240dbf97f080',
 '807b8f1c-6b6c-4a23-a73b-2247d05acda0']

****Retrieval Phase****

In [7]:
results = vector_store.similarity_search(
    "What is the revenue increase in FY24?",
    k=1
)

HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [8]:
results

[Document(id='807b8f1c-6b6c-4a23-a73b-2247d05acda0', metadata={'author': '(anonymous)', 'keywords': '', 'creationdate': '2025-12-14T08:45:46+00:00', 'title': '(anonymous)', 'trapped': '/False', 'moddate': '2025-12-14T08:45:46+00:00', 'page_label': '2', 'creator': '(unspecified)', 'total_pages': 2, 'subject': '(unspecified)', 'producer': 'ReportLab PDF Library - www.reportlab.com', 'page': 1, 'source': './TempData\\Acme_FY2024_UltraDense_Report.pdf'}, page_content='BALANCE SHEET, COMPLIANCE & RISK DISCLOSURES\nBALANCE SHEET SUMMARY (FY2024): Total assets amounted to USD 200.0 million, comprising current assets of USD 50.0\nmillion and non-current assets of USD 150.0 million. Total liabilities stood at USD 140.0 million, including short-term borrowings of\nUSD 32.0 million and long-term debt of USD 80.0 million. Total equity attributable to shareholders amounted to USD 60.0 million.\nBalance Sheet (USD)\nFY2024\nCash & Equivalents\n18,000,000\nAccounts Receivable\n22,000,000\nInventory\n